In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parents[0]))

import pandas as pd 
import numpy as np

from config import PROCESSED_DATA_DIR

In [2]:
reviews_df = pd.read_parquet(PROCESSED_DATA_DIR / "amazon_reviews_clean.parquet")

In [3]:
# One-hot encode country while retaining the original country column for traceability
# Top 10 most frequent countries will be their own category, less frequently appearing countries will be classified into OTHER
top_countries = reviews_df["country"].value_counts().head(10).index
reviews_df["country_grouped"] = reviews_df["country"].apply(
    lambda x: x if x in top_countries else "OTHER"
)

country_dummies = pd.get_dummies(reviews_df["country_grouped"], prefix="country")
reviews_df = pd.concat([reviews_df, country_dummies], axis=1)

country_dummies.head()

,country_AU,country_CA,country_DE,country_DK,country_GB,country_IE,country_IN,country_IT,country_NL,country_OTHER,country_US
0,False,False,False,False,False,False,False,False,False,False,True
1,False,False,False,False,True,False,False,False,False,False,False
2,False,False,False,False,True,False,False,False,False,False,False
3,True,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,True,False,False,False,False,False,False


In [4]:
# Combine review title and review text into a single feature
reviews_df["full_text"] = reviews_df["review_title"] + " " + reviews_df["review_text"]

In [5]:
# Log-transform review count to mitigate right-skewness
reviews_df["log_review_count"] = np.log1p(reviews_df["review_count"])

In [6]:
# Create text_length feature 
reviews_df["text_length"] = reviews_df["full_text"].str.len()

In [7]:
# Final check of engineered dataset
print(reviews_df.shape)
print(reviews_df.columns)
print(reviews_df.dtypes)

(21055, 22)
Index(['country', 'review_count', 'review_date', 'rating', 'review_title',
       'review_text', 'date_of_experience', 'country_grouped', 'country_AU',
       'country_CA', 'country_DE', 'country_DK', 'country_GB', 'country_IE',
       'country_IN', 'country_IT', 'country_NL', 'country_OTHER', 'country_US',
       'full_text', 'log_review_count', 'text_length'],
      dtype='str')
country                               str
review_count                        int64
review_date           datetime64[us, UTC]
rating                              int64
review_title                          str
review_text                           str
date_of_experience    datetime64[us, UTC]
country_grouped                       str
country_AU                           bool
country_CA                           bool
country_DE                           bool
country_DK                           bool
country_GB                           bool
country_IE                           bool
country_IN      

In [8]:
# Save engineered dataset
reviews_df.to_parquet(PROCESSED_DATA_DIR / "amazon_reviews_features.parquet", index=False)